## Step 1: Import Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from delta.tables import *
from delta.tables import DeltaTable


Incremental Dataset Path

In [0]:
incremental_path="/Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental"

##Step1:Read Silver Product

In [0]:
silver_products = spark.table("silver_products")

display(silver_products.limit(10))

product_id,product_name,category,brand,unit_price,status,created_date,ingesttime,source_file,load_type
P00003,Bedsheet 3,Home,BrandD,30753.26,active,2024-10-23,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00004,Oil 4,Grocery,BrandB,68176.44,active,2024-03-07,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00005,Oil 5,Unknown,BrandC,51796.39,active,2024-12-12,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00006,Chair 6,Home,BrandC,10740.11,discontinued,2025-06-17,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00007,Jeans 7,Fashion,BrandC,26020.02,active,2024-04-19,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00008,Chair 8,Home,BrandB,8710.16,active,2023-03-23,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00009,Cream 9,Beauty,BrandA,71925.36,discontinued,2023-02-22,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00010,Chair 10,Home,BrandD,25422.83,discontinued,2025-12-02,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00011,Jacket 11,Fashion,BrandA,26601.15,active,2025-06-29,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch
P00012,Tea 12,Grocery,BrandB,23734.04,active,2023-08-27,2026-07-12T01:00:10.731Z,dbfs:/Volumes/dbacademy/default/data/retail_delta_project/datasets/batch/products_batch.csv,batch


##Step2:Read CDC File

In [0]:
product_cdc = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{incremental_path}/day_2026-04-24/products_cdc_2026-04-24.csv")

display(product_cdc.limit(10))

product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation
P00435,Bedsheet 435,Home,BrandB,34748.71,active,2023-12-16,2026-04-24,UPDATE
P00140,Mouse 140,Electronics,BrandB,13985.55,active,2025-02-21,2026-04-24,UPDATE
P00081,Shampoo 81,Beauty,BrandD,73001.3,discontinued,2025-10-01,2026-04-24,UPDATE
P00561,Chair 561,Home,BrandC,45448.0,active,2025-10-14,2026-04-24,UPDATE
P00187,Rice 187,Grocery,BrandD,9702.61,active,2025-09-18,2026-04-24,UPDATE
P00096,Perfume 96,Beauty,BrandB,56592.01,active,2024-02-13,2026-04-24,UPDATE
P00089,Mouse 89,Electronics,BrandD,13704.24,active,2023-01-07,2026-04-24,UPDATE
P00005,Oil 5,null,BrandC,74867.82,active,2024-12-12,2026-04-24,UPDATE
P00183,Chair 183,Home,BrandB,23206.75,active,2024-08-21,2026-04-24,UPDATE
P00444,Mixer 444,Home,BrandC,22544.37,discontinued,2024-08-15,2026-04-24,UPDATE


In [0]:
product_cdc.select(
    *[
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in product_cdc.columns
    ]
).display()

product_id,product_name,category,brand,unit_price,status,created_date,effective_date,operation
0,0,3,0,0,0,0,0,0


In [0]:
product_cdc.printSchema()

print("No of incremental record:",product_cdc.count())

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- operation: string (nullable = true)

No of incremental record: 60


##Step3:Create Initial Dimension Table 

In [0]:
window = Window.orderBy("product_id")

dim_product = silver_products\
    .withColumn("product_sk", row_number().over(window))\
    .withColumn("effective_date", current_date())\
    .withColumn("end_date", lit(None).cast("date"))\
    .withColumn("is_current", lit(True))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


###3.1:Save Intial Dimension Table

In [0]:
dim_product.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("dim_products_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print(spark.table("dim_products_scd2").count())

780


In [0]:
product_cdc.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- operation: string (nullable = true)



In [0]:
spark.table("dim_products_scd2").printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- ingesttime: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- load_type: string (nullable = true)
 |-- product_sk: integer (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- is_current: boolean (nullable = true)



### 3.2:Separate records to be update and insert

#### Remove Duplicates 

In [0]:
window = Window.partitionBy("product_id", "operation").orderBy(desc("effective_date"))


product_update = product_cdc
    .filter(col("operation") == "UPDATE")\
    .withColumn("rn", row_number().over(window))\
    .filter("rn = 1")\
    .drop("rn")



In [0]:
product_insert = product_cdc\
    .filter(col("operation") == "INSERT")\
    .withColumn("rn", row_number().over(window))\
    .filter("rn = 1")
    .drop("rn")


In [0]:
print("Update Records :", product_update.count())
print("Insert Records :", product_insert.count())

Update Records : 60
Insert Records : 0


###3.3:Close Update Records

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "dim_products_scd2")


    deltaTable.alias("target").merge(
        product_update.alias("source"),
        "target.product_id = source.product_id AND target.is_current = true"
    )\
    .whenMatchedUpdate(
        set={
            "is_current": lit(False),
            "end_date": expr("date_sub(source.effective_date,1)")
        }
    )\
    .execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Total Products:",spark.table("dim_products_scd2").count())

print("Historical Products:", spark.table("dim_products_scd2")\
      .filter("is_current = false")
      .count())

Total Products: 780
Historical Products: 59


###3.4:Insert new update records

In [0]:
max_sk = spark.table("dim_products_scd2")\
    .agg(max("product_sk"))\
    .collect()[0][0]




In [0]:
window = Window.orderBy("product_id")

updated_products = product_update\
    .join(
        spark.table("dim_products_scd2").select("product_id"),
        "product_id",
        "inner"          
    )
    .dropDuplicates(["product_id"])
    .withColumn("product_sk", row_number().over(window) + max_sk)
    .withColumn("ingesttime", current_timestamp())
    .withColumn("source_file", lit("products_cdc_2026-04-24.csv"))
    .withColumn("load_type", lit("INCREMENTAL"))
    .withColumn("end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##Step4:Save Intial Products Dimension Table

In [0]:
updated_products.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "unit_price",
    "status",
    "created_date",
    "ingesttime",
    "source_file",
    "load_type",
    "product_sk",
    "effective_date",
    "end_date",
    "is_current"
).write \
.format("delta") \
.mode("append") \
.saveAsTable("dim_products_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print("Total:",spark.table("dim_products_scd2").count())

print("Current:",spark.table("dim_products_scd2")\
      .filter("is_current=true")\
      .count())

print("Historical:",spark.table("dim_products_scd2")\
      .filter("is_current=false")\
      .count())

Total: 839
Current: 780
Historical: 59


# Day-25 Incremental Records

In [0]:
product_cdc = spark.read
    .option("header", True)\
    .option("inferSchema", True)\
    .csv(f"{incremental_path}/day_2026-04-25/products_cdc_2026-04-25.csv")


In [0]:
window = Window.partitionBy("product_id", "operation").orderBy(desc("effective_date"))

product_update = product_cdc\
    .filter(col("operation") == "UPDATE")\
    .withColumn("rn", row_number().over(window))\
    .filter("rn = 1")
    .drop("rn")



In [0]:
product_insert =  product_cdc\
    .filter(col("operation") == "INSERT")\
    .withColumn("rn", row_number().over(window))\
    .filter("rn = 1")
    .drop("rn")


In [0]:
print("Update Records :", product_update.count())
print("Insert Records :", product_insert.count())

Update Records : 60
Insert Records : 0


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "dim_products_scd2")


    deltaTable.alias("target")
    .merge(
        product_update.alias("source"),
        "target.product_id = source.product_id AND target.is_current = true"
    ).whenMatchedUpdate(
        set={
            "is_current": lit(False),
            "end_date": expr("date_sub(source.effective_date,1)")
        }
    ).execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
print("Total Products:",spark.table("dim_products_scd2").count())

print("Historical Products:",spark.table("dim_products_scd2")\
      .filter("is_current = false")\
      .count())

Total Products: 839
Historical Products: 117


In [0]:
max_sk = spark.table("dim_products_scd2")\
    .agg(max("product_sk"))\
    .collect()[0][0]




In [0]:
window = Window.orderBy("product_id")

updated_products = product_update\
    .join(
        spark.table("dim_products_scd2").select("product_id"),
        "product_id",
        "inner"         
    )\
    .dropDuplicates(["product_id"])
    .withColumn("product_sk", row_number().over(window) + max_sk)
    .withColumn("ingesttime", current_timestamp())
    .withColumn("source_file", lit("products_cdc_2026-04-24.csv"))
    .withColumn("load_type", lit("INCREMENTAL"))
    .withColumn("end_date", lit(None).cast("date"))
    .withColumn("is_current", lit(True))



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
updated_products.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "unit_price",
    "status",
    "created_date",
    "ingesttime",
    "source_file",
    "load_type",
    "product_sk",
    "effective_date",
    "end_date",
    "is_current"
).write \
.format("delta") \
.mode("append") \
.saveAsTable("dim_products_scd2")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print("Total:",spark.table("dim_products_scd2").count())

print("Current:",spark.table("dim_products_scd2")\
      .filter("is_current=true")\
      .count())

print("Historical:",spark.table("dim_products_scd2")\
      .filter("is_current=false")\
      .count())

Total: 897
Current: 780
Historical: 117


#Function for incremental load(SCD Type 2)

In [0]:
def process_customer_cdc(cdc_path):

   
    product_cdc = spark.read\
        .option("header", True)\
        .option("inferSchema", True)\
        .csv(cdc_path)
    

    window = Window.partitionBy("product_id", "operation") \
        .orderBy(desc("effective_date"))

   product_update = product_cdc\
        .filter(col("operation") == "UPDATE")\
        .withColumn("rn", row_number().over(window))\
        .filter("rn = 1")
        .drop("rn")
    
    product_insert = product_cdc
        .filter(col("operation") == "INSERT")\
        .withColumn("rn", row_number().over(window))\
        .filter("rn = 1")
        .drop("rn")

    from delta.tables import DeltaTable

    deltaTable = DeltaTable.forName(spark, "dim_products_scd2")

    
        deltaTable.alias("target")
        .merge(
            product_update.alias("source"),
            "target.product_id = source.product_id AND target.is_current = true"
        ).whenMatchedUpdate(
            set={
                "is_current": lit(False),
                "end_date": expr("date_sub(source.effective_date,1)")
            }
        ).execute()
    

    max_sk = spark.table("dim_products_scd2")\
        .agg(max("product_sk"))\
        .collect()[0][0]
    

    window = Window.orderBy("product_id")

    updated_products = product_update
        .join(
            spark.table("dim_products_scd2").select("product_id"),
            "product_id",
            "inner"
        )
        .dropDuplicates(["product_id"])
        .withColumn("product_sk", row_number().over(window) + max_sk)
        .withColumn("ingesttime", current_timestamp())
        .withColumn("source_file", lit("products_cdc_2026-04-24.csv"))
        .withColumn("load_type", lit("INCREMENTAL"))
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
    
updated_products.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "unit_price",
    "status",
    "created_date",
    "ingesttime",
    "source_file",
    "load_type",
    "product_sk",
    "effective_date",
    "end_date",
    "is_current"
).write \
.format("delta") \
.mode("append") \
.saveAsTable("dim_products_scd2")

    print("Completed :", cdc_path)

In [0]:
process_customer_cdc("/Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental/day_2026-04-26/products_cdc_2026-04-26.csv")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Completed : /Volumes/dbacademy/default/data/retail_delta_project/datasets/incremental/day_2026-04-26/products_cdc_2026-04-26.csv


# Day-26 Incremental Records

In [0]:
print("Total:",spark.table("dim_products_scd2").count())

print("Current:",spark.table("dim_products_scd2")\
      .filter("is_current=true")\
      .count())

print("Historical:", spark.table("dim_products_scd2")\
      .filter("is_current=false")\
      .count())

Total: 955
Current: 780
Historical: 175
